# ಪಾಠ 08 - ಬಹು-ಏಜೆಂಟ್ ವಿನ್ಯಾಸ ಮಾದರಿ


## ಸೆಟಪ್


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

import os
import asyncio
import dotenv

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ಬಹು-ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳು ಏಕೆ?

ಪ್ರಯಾಣ ಯೋಜನೆ जैसी ವಾಸ್ತವವಲ್ಲದ ಕೆಲಸಗಳು ಬಹಳ ವಿಭಿನ್ನ ರೀತಿಯ ಪರಿಣತಿಗಳನ್ನು ಒಳಗೊಂಡಿರುತ್ತವೆ — ಸರಕಲು ಸಂಚಾರ, ಸ್ಥಳೀಯ ಜ್ಞಾನ, ಬಜೆಟಿಂಗ್ ಮತ್ತು ಇನ್ನಷ್ಟು. ಎಲ್ಲವನ್ನೂ ನಿಭಾಯಿಸಲು ಪ್ರಯತ್ನಿಸುವ ಒಂದು ಏಜೆಂಟ್ ಆಗಲೇದು ಹತಾಶವಾಗಬಹುದು.

ಬಹು-ಏಜೆಂಟ್ ವ್ಯವಸ್ಥೆಗಳು ಇದಕ್ಕೆ **ವಿಶೇಷೀಕರಣ** ಮೂಲಕ ಪರಿಹಾರ ನೀಡುತ್ತವೆ: ಪ್ರತಿ ಏಜೆಂಟ್ ತನ್ನ ಗ್ರಾಮದ ಪರಿಣತಿಯನ್ನು ಆವರಿಸಿಕೊಂಡು, ಸಾಮಾನ್ಯ ವ್ಯಕ್ತಿಗಿಂತ ಹೆಚ್ಚಿನ ಗುಣಮಟ್ಟದ ಫಲಿತಾಂಶಗಳನ್ನು ತರುತ್ತದೆ. ಇವರು **ವಿಸ್ತರಣಾರ್ಹತೆ** ಕೂಡ ಸುಧಾರಿಸುತ್ತಾರೆ — ನೀವು ಹೊಸ ಏಜೆಂಟ್‌ಗಳನ್ನು (ಉದಾಹರಣೆಗಾಗಿ, ವಿಮಾನ ತಜ್ಞ, ರೆಸ್ಟೋರೆಂಟ್ ವಿಮರ್ಶಕ) ಇದ್ದ ವ್ಯವಹಾರವನ್ನು ಮರುಬರೆಯದೆ ಸೇರಿಸಬಹುದು. ಏಜೆಂಟ್‌ಗಳು ಒಂದು ಸಂರಚಿತ ಪೈಪ್ಲೈನ್ ಮೂಲಕ ಪರಸ್ಪరం ಸಂಯೋಜನೆಯಾಗುತ್ತವೆ, ಒಂದರಿಂದ ಮತ್ತೊಬ್ಬಗೆ ಸಂದರ್ಭವನ್ನು ಬದಲಾಯಿಸುತ್ತವೆ.


## ವಿಶೇಷ ಏಜೆಂಟ್‌ಗಳನ್ನು ಸೃಷ್ಟಿಸುವುದು


In [ ]:
planner_agent = client.as_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = client.as_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## ಕ್ರಮಬದ್ಧ ಕಾರ್ಯಪ್ರವಾಹ ನಿರ್ಮಾಣ

`WorkflowBuilder` ನಿಮಗೆ ಏಜೆಂಟ್‌ಗಳನ್ನು ನಿರ್ದೇಶಿತ ಗ್ರಾಫ್ ಆಗಿ ಕನೆಕ್ಟ್ ಮಾಡಲು ನೆರವಾಗುತ್ತದೆ. ಇಲ್ಲಿ ನಾವು ಸರಳ ಎರಡು ಹಂತಗಳ ಪೈಪ್ಲೈನ್ ರಚಿಸುತ್ತೇವೆ: **TravelPlanner** ಪ್ರಯಾಣ ಯೋಜನೆಯನ್ನು ರಚಿಸುವುದು, ನಂತರ **TravelConcierge** ಅದನ್ನು ಪರಿಶೀಲಿಸಿ ಸುಧಾರಿಸುತ್ತದೆ.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ವರ್ಕ್ಫ್ಲೋಕ್ಕೆ ಇನ್ನೂ ಹೆಚ್ಚು ಏಜೆಂಟುಗಳನ್ನು ಸೇರಿಸಲಾಗುತ್ತಿದೆ

ಬಹು-ಏಜೆಂಟ್ ನಕ್ಷೆ ಯಲ್ಲಿನ ದೊಡ್ಡ ಲಾಭಗಳಲ್ಲಿ ಒಂದಾವುದು ಇದನ್ನು ವಿಸ್ತರಿಸುವುದು ಎಷ್ಟು ಸುಲಭವಾಗಿದೆ. ಕೆಳಗೆ ನಾವು ಒಂದು **BudgetReviewer** ಏಜೆಂಟ್ ಅನ್ನು ಸೇರಿಸಿದ್ದೇವೆ, ಅದು ಪ್ರಯಾಣಿಕರ ಬಜೆಟ್ಟಿಗೆ ಯೋಜನೆಯನ್ನು ಪರಿಶೀಲಿಸುತ್ತದೆ, ಲಿಮಿಟ್ ಮೀರಿದ ಕೊಷ್ಟುಗಳನ್ನು ತಳ್ಳಬಹುದಾದ ವಸ್ತುಗಳನ್ನು ಗುರುತಿಸುತ್ತದೆ ಮತ್ತು ಹಣ ಉಳಿತಾಯ ಮಾಡುವ ಪರ್ಯಾಯಗಳನ್ನು ಸಲಹೆ ನೀಡುತ್ತದೆ. ವರ್ಕ್ಫ್ಲೋ ಈಗ ಮೂರು ಏಜಂಟುಗಳನ್ನು ಕ್ರಮವಾಗಿ ನಡೆಸುತ್ತದೆ:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = client.as_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಹೇಗೆ ಕಲಿತೀರಿ:

1. **ವಿಶೇಷ ಕಾರ್ಯಕ್ರಮಗಳನ್ನು ರಚಿಸುವುದು** — ಪ್ರತಿಯೊಂದುಡುವು ನಿಗದಿತ ಪಾತ್ರವನ್ನು ಹೊಂದಿರುವುದು (ತಯಾರಿ, ಸಹಾಯಕ, ಬಜೆಟ್ ಪರಿಶೀಲನೆ).
2. **ಕಾರ್ಯಪ್ರವಾಹಕ್ಕೆ ಕಾರ್ಯಕ್ರಮಗಳನ್ನು ಸರಣಿಯಾಗಿ ಸಂಪರ್ಕಿಸುವುದು** `WorkflowBuilder` ಮತ್ತು `add_edge` ಬಳಸಿ.
3. **ಮಲ್ಟಿ-ಏಜೆಂಟ್ ಪೈಪ್‌ಲೈನ್‌ನಿಂದ ಔಟ್ಪುಟ್ ಅನ್ನು ಸ್ಟ್ರೀಮ್ ಮಾಡುವುದು**, ಯಾವ ಏಜೆಂಟ್ ಮಾತಾಡುತ್ತಿದೆ ಎಂದು ಟ್ರ್ಯಾಕ್ ಮಾಡುವುದು.
4. **ಕಾರ್ಯಪ್ರವಾಹವನ್ನು ವಿಸ್ತರಿಸುವುದು** ಈಗಿರುವ ಏಜೆಂಟ್‌ಗಳನ್ನು ಬದಲಾಯಿಸದೆ ಹೊಸ ಕಾರ್ಯಕ್ರಮಗಳನ್ನು ಸರಣಿಗೆ ಸೇರಿಸುವುದು.

ಮಲ್ಟಿ-ಏಜೆಂಟ್ ವಿನ್ಯಾಸ ಮಾದರಿ ಪ್ರತಿ ಏಜೆಂಟ್ ಅನ್ನು ಸರಳವಾಗಿಡುತ್ತದೆ, ಆದರೆ ಯಾವುದೇ ಒ ಎಂದಾದರೂ ಏಜೆಂಟ್ ಕಂಡಷ್ಟು ಅಲ್ಲದ, ಸಮೃದ್ಧ, ಹೆಚ್ಚು ಪರಿಶೀಲಿಸಿದ ಫಲಿತಾಂಶಗಳನ್ನು ಉಂಟುಮಾಡುತ್ತದೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
